# Model 5: Transformer U-Net (TransUNet-style) — Fetal Head Segmentation
**Dataset:** HC18 Grand Challenge — https://zenodo.org/records/1322001

**Architecture:** A hybrid **CNN + Vision Transformer** segmentation model inspired by TransUNet. The CNN encoder extracts local features, then a lightweight **Multi-Head Self-Attention (MHSA) transformer block** processes the bottleneck to capture long-range global dependencies. The decoder uses standard U-Net upsampling with skip connections.

**Why Transformers for fetal US?** The fetal head is an elliptical structure that spans a large spatial extent. CNNs have limited receptive fields in early layers — transformers at the bottleneck can model the relationship between distant skull boundary points, improving contour completeness.

**Implemented without external libraries** — uses PyTorch's built-in `nn.MultiheadAttention`.

## 1. Install & Download

In [ ]:
!pip install albumentations opencv-python matplotlib pandas tqdm einops -q
!wget -q --show-progress 'https://zenodo.org/records/1322001/files/training_set.zip' -O training_set.zip
!wget -q --show-progress 'https://zenodo.org/records/1322001/files/test_set.zip' -O test_set.zip
!unzip -q training_set.zip -d hc18
!unzip -q test_set.zip -d hc18
print('Ready.')

## 2. Imports & Config

In [ ]:
import os, random, math, time, warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from glob import glob

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

warnings.filterwarnings('ignore')
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

IMG_SIZE=256; BATCH_SIZE=8; EPOCHS=100; LR=1e-4; PATIENCE=15
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME='transformer_unet'
TRAIN_IMG_DIR='hc18/training_set'

print(f'Device: {DEVICE} | Model: Transformer U-Net (CNN + MHSA)')

## 3. Dataset

In [ ]:
def get_file_pairs(d):
    imgs=sorted(glob(os.path.join(d,'*_HC.png')))
    return [(i,i.replace('_HC.png','_HC_Annotation.png')) for i in imgs if os.path.exists(i.replace('_HC.png','_HC_Annotation.png'))]

def patient_split(pairs,val=0.15,test=0.10):
    rng=random.Random(SEED); p=pairs.copy(); rng.shuffle(p)
    n=len(p); nv,nt=int(n*val),int(n*test)
    return p[nt+nv:],p[nt:nt+nv],p[:nt]

def get_transforms(split):
    if split=='train':
        return A.Compose([A.PadIfNeeded(IMG_SIZE,IMG_SIZE),A.RandomCrop(IMG_SIZE,IMG_SIZE),
            A.HorizontalFlip(p=0.5),A.VerticalFlip(p=0.5),A.RandomRotate90(p=0.5),
            A.GaussNoise(p=0.3),A.RandomBrightnessContrast(p=0.3),
            A.Normalize(mean=(0.485,0.456,0.406),std=(0.229,0.224,0.225)),ToTensorV2()])
    return A.Compose([A.Resize(IMG_SIZE,IMG_SIZE),
        A.Normalize(mean=(0.485,0.456,0.406),std=(0.229,0.224,0.225)),ToTensorV2()])

class HC18Dataset(Dataset):
    def __init__(self,pairs,tfm=None):
        self.pairs=pairs; self.tfm=tfm
    def __len__(self): return len(self.pairs)
    def __getitem__(self,idx):
        ip,mp=self.pairs[idx]
        img=cv2.cvtColor(cv2.imread(ip),cv2.COLOR_BGR2RGB)
        msk=(cv2.imread(mp,cv2.IMREAD_GRAYSCALE)>127).astype(np.float32)
        if self.tfm:
            a=self.tfm(image=img,mask=msk); img,msk=a['image'],a['mask']
        return img, msk.unsqueeze(0)

all_pairs=get_file_pairs(TRAIN_IMG_DIR)
train_pairs,val_pairs,test_pairs=patient_split(all_pairs)
print(f'Train: {len(train_pairs)} | Val: {len(val_pairs)} | Test: {len(test_pairs)}')

train_dl=DataLoader(HC18Dataset(train_pairs,get_transforms('train')),BATCH_SIZE,shuffle=True, num_workers=0,pin_memory=True)
val_dl  =DataLoader(HC18Dataset(val_pairs,  get_transforms('val')),  BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
test_dl =DataLoader(HC18Dataset(test_pairs, get_transforms('val')),  BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)

## 4. Transformer U-Net Architecture

```
Input (B,3,256,256)
    │
  CNN Encoder (4 stages, feature maps: 64→128→256→512)
    │  skip connections s1,s2,s3,s4
  TransformerBottleneck
    ├── Flatten spatial dims → sequence of tokens
    ├── Add positional encoding
    ├── N × TransformerBlock (MHSA + FFN + LayerNorm)
    └── Reshape back to spatial feature map
    │
  CNN Decoder (4 stages with skip connections)
    │
  Output (B,1,256,256) → Sigmoid
```

**TransformerBlock:** Multi-Head Self-Attention + Feed-Forward Network, both with residual connections and pre-LayerNorm (more stable training than post-norm).

In [ ]:
# ── Building Blocks ───────────────────────────────────────────────────────────

class ConvBnRelu(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.net(x)


class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(ConvBnRelu(in_ch, out_ch), ConvBnRelu(out_ch, out_ch))
        self.pool = nn.MaxPool2d(2)
    def forward(self, x):
        skip = self.conv(x)
        return self.pool(skip), skip


class TransformerBlock(nn.Module):
    """
    Pre-LN Transformer block:
        x = x + MHSA(LN(x))
        x = x + FFN(LN(x))
    """
    def __init__(self, d_model, n_heads, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        mlp_dim    = int(d_model * mlp_ratio)
        self.ffn   = nn.Sequential(
            nn.Linear(d_model, mlp_dim), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, d_model), nn.Dropout(dropout)
        )

    def forward(self, x):
        # x: (B, N, d_model)
        ln  = self.norm1(x)
        attn_out, _ = self.attn(ln, ln, ln)
        x   = x + attn_out
        x   = x + self.ffn(self.norm2(x))
        return x


class TransformerBottleneck(nn.Module):
    """
    Flattens CNN feature map to sequence, runs N transformer blocks,
    reshapes back to spatial map.
    """
    def __init__(self, in_ch, d_model=512, n_heads=8, n_layers=6, dropout=0.1):
        super().__init__()
        self.in_ch   = in_ch
        self.d_model = d_model
        self.proj_in = nn.Conv2d(in_ch, d_model, 1, bias=False)  # channel projection
        self.pos_enc = None      # built lazily
        self.blocks  = nn.Sequential(*[
            TransformerBlock(d_model, n_heads, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.norm     = nn.LayerNorm(d_model)
        self.proj_out = nn.Conv2d(d_model, in_ch, 1, bias=False)

    def _build_pos_enc(self, h, w, device):
        """Learnable 2D positional encoding."""
        n = h * w
        return nn.Parameter(torch.zeros(1, n, self.d_model, device=device))

    def forward(self, x):
        B, C, H, W = x.shape
        x  = self.proj_in(x)                                # (B, d, H, W)
        x  = x.flatten(2).transpose(1, 2)                  # (B, H*W, d)

        # Add learnable position encoding
        if self.pos_enc is None or self.pos_enc.shape[1] != H*W:
            self.pos_enc = self._build_pos_enc(H, W, x.device)
        x = x + self.pos_enc

        for blk in self.blocks:
            x = blk(x)                                      # (B, N, d)
        x  = self.norm(x)
        x  = x.transpose(1, 2).reshape(B, self.d_model, H, W)  # back to spatial
        return self.proj_out(x)                             # (B, C, H, W)


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = nn.Sequential(ConvBnRelu(out_ch+skip_ch, out_ch), ConvBnRelu(out_ch, out_ch))
    def forward(self, x, skip):
        return self.conv(torch.cat([self.up(x), skip], dim=1))


class TransformerUNet(nn.Module):
    def __init__(self, in_ch=3,
                 features=[64, 128, 256, 512],
                 d_model=512, n_heads=8, n_layers=4):
        super().__init__()
        # CNN Encoder
        self.enc = nn.ModuleList()
        prev = in_ch
        for f in features:
            self.enc.append(EncoderBlock(prev, f)); prev=f

        # Transformer Bottleneck
        self.transformer = TransformerBottleneck(
            in_ch=features[-1], d_model=d_model,
            n_heads=n_heads, n_layers=n_layers
        )

        # CNN Decoder
        rev = list(reversed(features))
        self.dec = nn.ModuleList()
        in_d = features[-1]
        for f in rev:
            self.dec.append(DecoderBlock(in_d, f, f)); in_d=f

        self.head = nn.Conv2d(features[0], 1, 1)

    def forward(self, x):
        skips = []
        for enc in self.enc:
            x, skip = enc(x); skips.append(skip)

        x = self.transformer(x)     # global context via MHSA

        for dec, skip in zip(self.dec, reversed(skips)):
            x = dec(x, skip)

        return torch.sigmoid(self.head(x))


# Use smaller d_model/n_layers if on CPU or low VRAM
d_model  = 512 if DEVICE == 'cuda' else 256
n_layers = 4   if DEVICE == 'cuda' else 2

model = TransformerUNet(d_model=d_model, n_layers=n_layers).to(DEVICE)
print(f'Transformer U-Net params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print(f'  Transformer: d_model={d_model}, n_heads=8, n_layers={n_layers}')

## 5. Loss, Metrics & Optimizer

In [ ]:
def hybrid_loss(pred, target, smooth=1e-6):
    bce  = F.binary_cross_entropy(pred, target)
    inter= (pred*target).sum(dim=(2,3))
    dice = 1-(2*inter+smooth)/(pred.sum(dim=(2,3))+target.sum(dim=(2,3))+smooth)
    return bce+dice.mean()

def compute_metrics(pred, target, thresh=0.5, smooth=1e-6):
    p=(pred>thresh).float()
    tp=(p*target).sum(); fp=(p*(1-target)).sum()
    fn=((1-p)*target).sum(); tn=((1-p)*(1-target)).sum()
    dice=(2*tp+smooth)/(2*tp+fp+fn+smooth)
    iou=(tp+smooth)/(tp+fp+fn+smooth)
    prec=(tp+smooth)/(tp+fp+smooth); rec=(tp+smooth)/(tp+fn+smooth)
    acc=(tp+tn+smooth)/(tp+tn+fp+fn+smooth)
    spec=(tn+smooth)/(tn+fp+smooth)
    return {'dice':dice.item(),'iou':iou.item(),'precision':prec.item(),'recall':rec.item(),
            'f1':dice.item(),'accuracy':acc.item(),'specificity':spec.item()}

# Use AdamW — better weight decay handling for transformer parameters
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
# Cosine annealing — widely used for transformers
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
print('Optimizer: AdamW | Scheduler: CosineAnnealing')

## 6. Training

In [ ]:
history={'train_loss':[],'val_loss':[],'val_dice':[],'val_iou':[]}
best_loss=float('inf'); pat=0

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    tloss=0; all_m={k:0.0 for k in ['dice','iou','precision','recall','f1','accuracy','specificity']}
    ctx=torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs,masks in tqdm(loader,leave=False):
            imgs,masks=imgs.to(DEVICE),masks.to(DEVICE)
            preds=model(imgs); loss=hybrid_loss(preds,masks)
            if train: optimizer.zero_grad(); loss.backward(); optimizer.step()
            tloss+=loss.item()
            m=compute_metrics(preds.detach(),masks)
            for k in all_m: all_m[k]+=m[k]
    n=len(loader)
    return tloss/n,{k:v/n for k,v in all_m.items()}

print(f'Training Transformer U-Net for {EPOCHS} epochs...\n')
start=time.time()

for epoch in range(1, EPOCHS+1):
    tl,_=run_epoch(train_dl,True); vl,vm=run_epoch(val_dl,False)
    scheduler.step()
    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['val_dice'].append(vm['dice']); history['val_iou'].append(vm['iou'])
    if epoch%10==0 or epoch==1:
        print(f'Ep {epoch:03d} | Train:{tl:.4f} | Val:{vl:.4f} | Dice:{vm["dice"]:.4f} | IoU:{vm["iou"]:.4f} | LR:{scheduler.get_last_lr()[0]:.2e}')
    if vl<best_loss:
        best_loss=vl; torch.save(model.state_dict(),f'best_{MODEL_NAME}.pth'); pat=0
    else:
        pat+=1
        if pat>=PATIENCE: print(f'Early stop ep {epoch}'); break

print(f'Done in {(time.time()-start)/60:.1f} min')

## 7. Training Curves

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4))
fig.suptitle('Transformer U-Net Training History')
axes[0].plot(history['train_loss'],label='Train'); axes[0].plot(history['val_loss'],label='Val')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(history['val_dice'],color='green'); axes[1].set_title('Val Dice')
axes[2].plot(history['val_iou'],color='orange'); axes[2].set_title('Val IoU')
for ax in axes: ax.set_xlabel('Epoch')
plt.tight_layout(); plt.savefig(f'{MODEL_NAME}_curves.png',dpi=150); plt.show()

## 8. Test & HC Estimation

In [ ]:
model.load_state_dict(torch.load(f'best_{MODEL_NAME}.pth', map_location=DEVICE))
test_loss, test_mets = run_epoch(test_dl, False)
print('='*50); print('  Transformer U-Net — Test Results'); print('='*50)
for k,v in test_mets.items(): print(f'  {k:<12}: {v*100:.2f}%')
print(f'  {"loss":<12}: {test_loss:.4f}'); print('='*50)

In [ ]:
def estimate_hc(mask_np,ps=0.143):
    u8=(mask_np*255).astype(np.uint8)
    u8=cv2.morphologyEx(u8,cv2.MORPH_CLOSE,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5)))
    cs,_=cv2.findContours(u8,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    if not cs: return None
    lc=max(cs,key=cv2.contourArea)
    if len(lc)<5: return None
    (_,_),(MA,ma),_=cv2.fitEllipse(lc)
    a,b=MA/2,ma/2
    return math.pi*(3*(a+b)-math.sqrt((3*a+b)*(a+3*b)))*ps

dm=torch.tensor([0.485,0.456,0.406]).view(3,1,1)
ds=torch.tensor([0.229,0.224,0.225]).view(3,1,1)
model.eval(); hc_errors=[]
fig,axes=plt.subplots(3,4,figsize=(16,12))

with torch.no_grad():
    for idx,(imgs,masks) in enumerate(test_dl):
        if idx>=1: break
        preds=model(imgs.to(DEVICE)).cpu()
        for i in range(min(3,imgs.shape[0])):
            inp=(imgs[i]*ds+dm).permute(1,2,0).numpy().clip(0,1)
            mnp=masks[i,0].numpy(); pnp=(preds[i,0]>0.5).numpy().astype(np.float32)
            hcp=estimate_hc(pnp); hcg=estimate_hc(mnp)
            axes[i][0].imshow(inp);              axes[i][0].set_title('Input')
            axes[i][1].imshow(mnp,cmap='gray');  axes[i][1].set_title('GT')
            axes[i][2].imshow(pnp,cmap='gray');  axes[i][2].set_title('Pred')
            ov=(inp*255).astype(np.uint8).copy()
            cs2,_=cv2.findContours((pnp*255).astype(np.uint8),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(ov,cs2,-1,(0,255,0),2)
            axes[i][3].imshow(ov); axes[i][3].set_title(f'HC={hcp:.1f}mm' if hcp else '—')
            if hcp and hcg: hc_errors.append(abs(hcp-hcg))
            for ax in axes[i]: ax.axis('off')

plt.suptitle('Transformer U-Net — Qualitative Results',fontsize=14)
plt.tight_layout(); plt.savefig(f'{MODEL_NAME}_qualitative.png',dpi=150); plt.show()
if hc_errors: print(f'HC MAE: {np.mean(hc_errors):.2f} mm | MSE: {np.mean(np.array(hc_errors)**2):.2f} mm²')

## 9. Visualize Attention — Where Does the Model Look?

In [ ]:
# Extract attention maps from the first transformer block for one test image
model.eval()

with torch.no_grad():
    for imgs, masks in test_dl:
        img = imgs[:1].to(DEVICE)  # single image
        
        # Hook to capture attention weights
        attn_weights = []
        def hook_fn(module, inp, out):
            # nn.MultiheadAttention returns (output, attn_weights)
            pass
        
        # Manually get attention from first block
        x = img
        for enc in model.enc:
            x, _ = enc(x)
        
        # Get feature map before transformer
        feat_map = model.transformer.proj_in(x)
        B, d, H, W = feat_map.shape
        tokens = feat_map.flatten(2).transpose(1, 2)  # (1, N, d)
        if model.transformer.pos_enc is not None:
            tokens = tokens + model.transformer.pos_enc
        
        ln = model.transformer.blocks[0].norm1(tokens)
        _, attn_map = model.transformer.blocks[0].attn(ln, ln, ln, average_attn_weights=True)
        # attn_map: (1, N, N) — attention from each token to all others
        
        # Average attention from all tokens → reshape to spatial
        attn_avg = attn_map[0].mean(0).reshape(H, W).cpu().numpy()
        attn_avg = (attn_avg - attn_avg.min()) / (attn_avg.max() - attn_avg.min() + 1e-8)
        attn_resized = cv2.resize(attn_avg, (IMG_SIZE, IMG_SIZE))
        
        img_np = (imgs[0] * ds + dm).permute(1,2,0).numpy().clip(0,1)
        
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(img_np); axes[0].set_title('Input'); axes[0].axis('off')
        axes[1].imshow(attn_resized, cmap='hot'); axes[1].set_title('Attention Map'); axes[1].axis('off')
        
        overlay = img_np.copy()
        heatmap = plt.cm.hot(attn_resized)[:, :, :3]
        blend = (0.6 * overlay + 0.4 * heatmap).clip(0, 1)
        axes[2].imshow(blend); axes[2].set_title('Overlay'); axes[2].axis('off')
        
        plt.suptitle('Transformer Self-Attention Visualization')
        plt.tight_layout()
        plt.savefig(f'{MODEL_NAME}_attention_map.png', dpi=150)
        plt.show()
        break

## 10. Save Results

In [ ]:
results={'model':'Transformer-UNet',**{k:round(v*100,2) for k,v in test_mets.items()},'test_loss':round(test_loss,4)}
if hc_errors:
    results['hc_mae_mm']=round(float(np.mean(hc_errors)),3)
    results['hc_mse_mm2']=round(float(np.mean(np.array(hc_errors)**2)),3)
df=pd.DataFrame([results])
df.to_csv(f'{MODEL_NAME}_results.csv',index=False)
print(df.to_string(index=False))

## 11. Compare All 5 Models (run after all notebooks complete)

In [ ]:
import glob

result_files = glob.glob('*_results.csv')
all_results = [pd.read_csv(f) for f in result_files]

if all_results:
    combined = pd.concat(all_results, ignore_index=True)
    combined = combined.sort_values('dice', ascending=False)
    combined.to_csv('all_models_comparison.csv', index=False)
    
    print('\n' + '='*80)
    print('  ALL MODELS COMPARISON (sorted by Dice)')
    print('='*80)
    print(combined[['model','dice','iou','precision','recall','f1','accuracy','test_loss']].to_string(index=False))
    print('='*80)
    
    # Bar chart comparison
    metrics = ['dice','iou','precision','recall','f1']
    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(combined))
    width = 0.15
    colors = ['#2196F3','#4CAF50','#FF9800','#E91E63','#9C27B0']
    
    for i, (m, c) in enumerate(zip(metrics, colors)):
        ax.bar(x + i*width, combined[m], width, label=m, color=c, alpha=0.85)
    
    ax.set_xticks(x + width*2)
    ax.set_xticklabels(combined['model'], rotation=15)
    ax.set_ylabel('Score (%)')
    ax.set_title('All 5 Models — Key Metrics Comparison')
    ax.legend(loc='lower right')
    ax.set_ylim(0, 105)
    plt.tight_layout()
    plt.savefig('all_models_comparison.png', dpi=150)
    plt.show()
    
    best = combined.iloc[0]
    print(f'\n🏆 Best model: {best["model"]} — Dice: {best["dice"]:.2f}% | IoU: {best["iou"]:.2f}%')
    print('→ This model is the best candidate to combine with MiT-B2 encoder from the base paper.')
else:
    print('Run all 5 notebooks first to collect results.')